# Recipe 05 — Pipeline Diagnostics
> Understand exactly what happened during a context build.

| | |
|---|---|
| **Module** | `anchor.pipeline` |
| **Key classes** | `PipelineDiagnostics`, `ContextWindow` |
| **Difficulty** | Beginner |

In [ ]:
from anchor.pipeline import ContextPipeline, PipelineStep
from anchor.pipeline.steps import retriever_step, filter_step
from anchor.models import QueryBundle, ContextItem, SourceType
from anchor.formatters import GenericTextFormatter
from anchor.memory import SlidingWindowMemory

## 1 — Build a pipeline with several steps
We need a multi-step pipeline to generate interesting diagnostics.

In [ ]:
class MockRetriever:
    def retrieve(self, query, top_k=5):
        return [
            ContextItem(
                id=f"doc-{i}", content=f"Result {i}",
                source=SourceType.RETRIEVAL,
                score=round(0.95 - i * 0.15, 2),
                token_count=10,
            )
            for i in range(top_k)
        ]

pipeline = ContextPipeline(max_tokens=4096)
pipeline.add_system_prompt("You are a helpful assistant.")
pipeline.with_formatter(GenericTextFormatter())

memory = SlidingWindowMemory(max_tokens=1024)
memory.add_turn("user", "previous question")
pipeline.with_memory(memory)

pipeline.add_step(retriever_step("fetch", retriever=MockRetriever(), top_k=8))
pipeline.add_step(filter_step(
    "quality",
    filter_fn=lambda items, q: [i for i in items if (i.score or 0) > 0.4],
))

query = QueryBundle(query_str="diagnostics demo")
result = pipeline.build(query)
print("Pipeline built")

## 2 — Access `result.diagnostics`
`diagnostics` is a dict-like `PipelineDiagnostics` object summarizing the
entire build.

In [ ]:
diag = result.diagnostics
print(f"Steps executed          : {diag.get('steps')}")
print(f"Total items considered  : {diag.get('total_items_considered')}")
print(f"Items included          : {diag.get('items_included')}")
print(f"Items overflowed        : {diag.get('items_overflow')}")
print(f"Memory items            : {diag.get('memory_items')}")

## 3 — Token utilization
See how efficiently the token budget was used.

In [ ]:
print(f"Token utilization : {diag.get('token_utilization', 0):.1%}")

usage = diag.get('token_usage_by_source', {})
for source, tokens in usage.items():
    print(f"  {source}: {tokens} tokens")

## 4 — Window-level metrics
`result.window` exposes raw numbers directly.

In [ ]:
window = result.window
print(f"Max tokens       : {window.max_tokens}")
print(f"Used tokens      : {window.used_tokens}")
print(f"Remaining tokens : {window.remaining_tokens}")
print(f"Utilization      : {window.utilization:.1%}")
print(f"Item count       : {len(window.items)}")

## 5 — Per-item inspection
Walk through included items to understand what made it in.

In [ ]:
for item in result.window.items:
    print(f"  {item.id:8s}  source={item.source}  "
          f"score={item.score}  tokens={item.token_count}")

## Key Takeaways
- `result.diagnostics` provides a build-level summary (steps, counts, utilization).
- `result.window` gives item-level detail (tokens, scores, priorities).
- Use diagnostics during development to tune retrieval and filtering parameters.

**Next:** [Context Window](06_context_window.ipynb)